# Milestone 9d - Ship, monitor, and grow the eval set

Part 4 of 4: the end of the loop, and the start of the next one.

1. **Ship** behind the must-pass gate, then a canary.
2. **Monitor** production for a signal that something changed.
3. **Trace the signal to a cause**, and fix it.
4. **Grow the eval set** with the new failures, as a new version, so the next
   experiment starts from a better set.


In **9c** the rule blocked v2 (its rewrite broke a must-pass question), and the
planned follow-up, **the reranker without the rewrite** (`v2-no-rewrite`), won on
dev and held on the holdout. That's the version shipping here.

**Do you need Ollama?** It's best with it, running `gemma3:12b`. Without it,
every model reply is replayed from a recording.

In [ ]:
import ipytest
import mlflow
import pandas as pd
import pytest
from mlflow.entities import AssessmentSource, SpanType
from mlflow.genai import datasets

from drill import rag_lab as R

ipytest.autoconfig(clean="test_*")

URI = R.use_mlflow("m9-help-center")
SPLIT = R.default_split()
SHIPPED = "v2-no-rewrite"   # the version 9c decided to run in production

lab = R.Lab()
print("Live: answers come from Ollama." if lab.live else "Offline: model replies are replayed from the recording.")

## 1. Ship behind the gate, then a canary

Two checks stand between a decision and real users:

- **The must-pass gate**, run in the release pipeline, not by hand. If any
  must-pass question fails, the release stops.
- **A canary**: send a small share of real traffic, say 10%, to the new version,
  watch the same numbers as before, and only then send everyone. The Daily Drill
  lesson "Canary releases" shows how the routing works.

One detail from 9c: v1 was **already** failing one must-pass question in
production. The gate's job is to stop *new* failures, so known ones are listed
explicitly, each with an owner and a fix planned, rather than silently ignored.

In [ ]:
KNOWN_FAILING = {"refund-01": "already failing in v1; keyword search can't find the return-window article"}

gate = lab.run([R.BY_ID[q] for q in R.MUST_PASS], [SHIPPED], runs=1, tags={"stage": "release-gate"}, quiet=True)
failing = [r["qid"] for r in gate if not r["passed"]]
new_failures = [q for q in failing if q not in KNOWN_FAILING]
print("known failures, tracked:", {q: KNOWN_FAILING[q] for q in failing if q in KNOWN_FAILING})
print("release gate:", "passed, safe to start the canary" if not new_failures else f"FAILED on {new_failures}: release stopped")

## 2. Monitor: a signal, not a score

In production nobody has labelled the answers, so you can't measure accuracy
directly. You watch **signals** that move when quality moves. A useful one for
a help-center assistant: how often it says **"I don't know"**, per group. A
sudden rise means people are asking about something it can't find.

A week after release, the billing team publishes a new article, "Pausing your
plan", and customers start asking about it. Here is the week's billing traffic:
the usual questions plus the new ones.

**Your turn.** Write `decline_rates(rows)`: `rows` is a list of
`(group, declined)` pairs. Return a dict from group to the share of its answers
that declined, rounded to 2 decimal places.

In [ ]:
def decline_rates(rows):
    """{group: share of answers that said "I don't know"}, rounded to 2 places."""
    # TODO: count answers and declines per group, then divide.
    raise NotImplementedError("Milestone 9d: implement decline_rates")

In [ ]:
%%ipytest -qq

def test_rates_per_group():
    rows = [("billing", True), ("billing", False), ("billing", False), ("billing", False), ("fraud", False)]
    assert decline_rates(rows) == {"billing": 0.25, "fraud": 0.0}


def test_rounding():
    assert decline_rates([("x", True), ("x", False), ("x", False)]) == {"x": 0.33}


def test_no_rows():
    assert decline_rates([]) == {}

In [ ]:
billing = [q for q in R.QUESTIONS if q["group"] == "billing"][:6]
before = lab.run(billing, [SHIPPED], runs=1, tags={"stage": "production", "week": "1"}, quiet=True)

lab.publish(R.NEW_KB, reindex=False)          # the article goes live... but nobody reruns the search indexing job
week2 = lab.run(R.NEW_QUESTIONS + billing, [SHIPPED], runs=1, tags={"stage": "production", "week": "2"}, quiet=True)

for name, batch in [("week 1", before), ("week 2", week2)]:
    print(name, decline_rates([(r["group"], R.declined(r["answer"])) for r in batch]))

**What happened.** On the recorded run, the share of billing answers saying "I
don't know" jumps from **17%** in week 1 to **40%** in week 2. Nothing in the app
changed, so something in the world did. That's the signal to go and look.

Notice what the signal **missed**, though. One new question, "How long can I take
a break from my subscription?", didn't decline: it confidently answered with the
auto-renew article, which is wrong. A decline rate can't see confident wrong
answers. That's why teams keep labelling a sample of production traffic after
release, as in 9a, instead of relying on signals alone.

## 3. Trace the signal to a cause

A signal says *something* changed. The traces say *what*. Take the week-2 answers
that declined and look at their **RETRIEVER** step: what did search find?

In [ ]:
declined_now = [r for r in week2 if R.declined(r["answer"])]
for r in declined_now:
    trace = mlflow.get_trace(r["trace_id"])
    found = [d["doc_id"] for d in trace.search_spans(span_type=SpanType.RETRIEVER)[0].outputs]
    print(f"{r['qid']:11} {R.BY_ID[r['qid']]['question'][:52]:52}  search found {found[:4]}")

print("\nis the new article in the knowledge base?", "kb-25" in lab.kb)
print("can search find it?", "kb-25" in lab.index.search("pause my plan", 10))

**What happened.** The new article, `kb-25`, is in the knowledge base, but search
can't find it: the search index was built before the article was published, and
nobody rebuilt it. Every question about pausing gets the old articles, so the
model correctly says it doesn't know. The traces showed it in one step; without
the RETRIEVER spans you'd be guessing.

In [ ]:
lab.publish({}, reindex=True)                 # the fix: rebuild the search index
fixed = lab.run(R.NEW_QUESTIONS, [SHIPPED], runs=1, tags={"stage": "production", "week": "2-fixed"}, quiet=True)
for r in fixed:
    print(f"{r['qid']:11} {'pass' if r['passed'] else 'FAIL'}   {r['answer'][:90]}")

**What happened.** Rebuilding the index fixes half of it. "Will I be charged while
my plan is paused?" and "Where's the setting to freeze my plan?" now pass. But
"put my subscription **on hold**" and "take a **break** from my subscription"
still fail: their words don't appear in the article, so keyword search can't
connect them to "pause". They're **out of reach**, exactly the kind of miss the
rewrite rescued in 9c, before it broke the refund question.

## 4. Grow the eval set

The new kind of question is now part of real traffic, so it belongs in the eval
set, or the next experiment can't see it. Same steps as 9a, on the new traces:
experts add **expectations**, the traces are **merged** in, and the new questions
are **dealt** into dev / test / holdout.

One subtlety. The existing split dealt each group with the pattern dev, dev,
dev, test, holdout. New billing questions should carry on that pattern from
where billing left off, not restart it, or the proportions drift.

**Your turn.** Write `deal_more(counts, new_ids_by_group)`: `counts` maps each
group to how many questions it already has; deal each group's new ids continuing
its pattern. Return a dict from new id to split.

In [ ]:
def deal_more(counts, new_ids_by_group):
    """Deal new ids 3-1-1, continuing each group's pattern from its current count."""
    # TODO: a group with `start` questions already gives its next new id pattern[(start + offset) % 5].
    raise NotImplementedError("Milestone 9d: implement deal_more")

In [ ]:
%%ipytest -qq

def test_continues_the_pattern():
    # billing already has 13: positions 13 and 14 are test and holdout, then it starts again at dev
    assert deal_more({"billing": 13}, {"billing": ["n1", "n2", "n3"]}) == {"n1": "test", "n2": "holdout", "n3": "dev"}


def test_a_new_group_starts_at_the_beginning():
    assert deal_more({}, {"pausing": ["p1", "p2"]}) == {"p1": "dev", "p2": "dev"}


def test_nothing_new():
    assert deal_more({"billing": 13}, {}) == {}

In [ ]:
EXPERT = AssessmentSource(source_type="HUMAN", source_id="support-expert")
for r in fixed:
    q = R.BY_ID[r["qid"]]
    mlflow.log_expectation(trace_id=r["trace_id"], name="expected_facts", value=q["facts"], source=EXPERT)
    mlflow.log_expectation(trace_id=r["trace_id"], name="expected_doc", value=q["gold"], source=EXPERT)

try:
    eval_set = datasets.get_dataset(name="support-eval")
except Exception:   # running 9d on its own: rebuild 9a's eval set first
    eval_set = datasets.create_dataset(name="support-eval")
    eval_set.merge_records([
        {"inputs": {"question": q["question"]},
         "expectations": {"expected_facts": q["facts"], "expected_doc": q["gold"] or "none"},
         "tags": {"qid": q["id"], "group": q["group"], "split": SPLIT[q["id"]]}} for q in R.QUESTIONS])

counts = {}
for q in R.QUESTIONS:
    counts[q["group"]] = counts.get(q["group"], 0) + 1
new_split = deal_more(counts, {"billing": [q["id"] for q in R.NEW_QUESTIONS]})

before_n = len(eval_set.to_df())
eval_set.merge_records([
    {"inputs": {"question": q["question"]},
     "expectations": {"expected_facts": q["facts"], "expected_doc": q["gold"]},
     "tags": {"qid": q["id"], "group": q["group"], "split": new_split[q["id"]], "added": "after-release"}}
    for q in R.NEW_QUESTIONS])
print(f"eval set: {before_n} -> {len(eval_set.to_df())} questions")
print("new questions dealt:", new_split)

**What happened.** The eval set grows from 55 to 59 questions, and the new ones
carry on billing's dev / test / holdout pattern where it left off. In real use you
would also:

- **Fix what reading regressions found.** 9c showed one expected answer was too
  strict ("before your order ships" is correct); update its expectations in the
  dataset, so the next comparison doesn't repeat the false alarm.
- **Add to the must-pass set** if a new question is critical. "Will I be charged
  while my plan is paused?" is about money, so it's a good candidate.
- **Keep the old version of the dataset.** Every merge creates a new version, so an
  old experiment can always be re-run on exactly the set it used.

## The loop, closed

You've now been through every stage once:

| Stage | Where |
|---|---|
| Collect, label, build the eval set | 9a |
| Frame, design, build and trace | 9b |
| Evaluate and decide | 9c |
| Ship, monitor, grow the eval set | 9d |

And it starts again: the next experiment begins from a bigger eval set, with a
must-pass set that includes what went wrong last time, and with ideas for what
to try next that came from reading the traces, not from guessing.


**The next experiment writes itself**, from evidence rather than guesses: a
**rewrite that doesn't break the refund question**. The eval set now contains the
two vocabulary-mismatch questions it needs to fix (from 9d) and the must-pass
question it must not break (from 9c). Write the plan first.